# 交互式映射与可视化

## 介绍

可视化在GeoAI工作流程的每个阶段都至关重要，从训练前检查图像和验证标签，到评估预测和传达结果。

静态绘图适用于简单检查，但地理空间数据需要交互性，以便您可以平移、缩放、切换图层和并排比较数据集。

[leafmap](https://leafmap.org)库提供了一个Pythonic接口，用于在Jupyter笔记本中构建交互式地图，并提供了加载栅格、矢量和云托管地理空间数据的便捷函数。

本教程涵盖核心可视化技术：创建交互式地图、添加栅格和矢量图层、构建分割面板比较，以及将模型预测叠加在源图像上。所有示例都使用拉斯维加斯建筑检测项目的数据集，包括NAIP航空图像、LiDAR衍生的地面高度（HAG）栅格、建筑轮廓标注和栅格化建筑掩码。

## 学习目标

在本教程结束时，您将能够：

- 使用leafmap创建交互式地图并自定义其外观
- 使用适当的颜色映射和波段组合将栅格数据（GeoTIFF、COG）添加到地图
- 使用自定义样式可视化矢量数据（GeoJSON、GeoDataFrame）
- 构建分割面板地图以比较数据集或时间段
- 将模型预测叠加在源图像上进行视觉评估
- 可视化Microsoft Planetary Computer的云托管数据

## 安装

取消注释以下行以安装所需的包。

In [ ]:
# %pip install "geoai-py[extra]"

## 使用Leafmap创建交互式地图

### 您的第一张地图

调用`leafmap.Map()`创建具有平移、缩放和图层控制的交互式地图小部件。您可以配置地图中心、缩放级别和显示尺寸。

In [1]:
import leafmap

m = leafmap.Map(center=[36.1617, -115.1524], zoom=11, height="600px")
m

Map(center=[36.1617, -115.1524], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', '…

`center`参数接受`[纬度, 经度]`对，`zoom`控制初始缩放级别，`height`设置小部件高度。

### 添加底图

底图提供地理背景，使地理空间可视化有意义。`leafmap`通过`add_basemap()`方法提供对数百个底图的访问。

In [2]:
m = leafmap.Map()
m.add_basemap("Esri.WorldImagery")
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

要查看所有可用的底图，请检查`leafmap.basemaps`字典：

In [3]:
basemaps = list(leafmap.basemaps.keys())
print(f"Total basemaps: {len(basemaps)}")
print("First 10:", basemaps[:10])

Total basemaps: 155
First 10: ['OpenStreetMap', 'Esri.WorldStreetMap', 'Esri.WorldImagery', 'Esri.WorldTopoMap', 'FWS NWI Wetlands', 'FWS NWI Wetlands Raster', 'NLCD 2021 CONUS Land Cover', 'NLCD 2019 CONUS Land Cover', 'NLCD 2016 CONUS Land Cover', 'NLCD 2013 CONUS Land Cover']


您也可以添加多个底图并使用图层控制在它们之间切换：

In [4]:
m = leafmap.Map()
m.add_basemap("Esri.WorldImagery")
m.add_basemap("OpenTopoMap")
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

## 在地图上处理栅格数据

栅格数据构成了大多数GeoAI工作流程的基础。`leafmap`提供了多种方法将栅格数据添加到地图，每种方法都针对不同的数据源和访问模式进行了优化。

### 添加GeoTIFF图层

`add_raster()`方法加载本地或远程GeoTIFF，并使用您选择的颜色映射在地图上显示它。在本教程中，我们使用拉斯维加斯建筑检测项目的四个数据集：

- **NAIP航空图像**（4波段GeoTIFF）：60厘米分辨率的红、绿、蓝和近红外波段
- **地面高度（HAG）**（单波段GeoTIFF）：LiDAR衍生的地表高度（米）
- **建筑轮廓**（GeoJSON）：勾勒单个建筑的矢量多边形
- **建筑掩码**（单波段GeoTIFF）：栅格化的二进制掩码，其中1表示建筑像素，0表示背景

In [5]:
import geoai

我们定义每个数据集的URL，并使用`geoai.download_file()`在本地获取它们：

In [26]:
naip_url = "https://data.source.coop/opengeos/geoai/las-vegas-train-naip.tif"
hag_url = "https://data.source.coop/opengeos/geoai/las-vegas-train-hag.tif"
buildings_url = (
    "https://data.source.coop/opengeos/geoai/las-vegas-buildings-train.geojson"
)
buildings_mask_url = (
    "https://data.source.coop/opengeos/geoai/las-vegas-buildings-mask.tif"
)

In [27]:
naip_path = geoai.download_file(naip_url)
hag_path = geoai.download_file(hag_url)
buildings_path = geoai.download_file(buildings_url)
mask_path = geoai.download_file(buildings_mask_url)

In [30]:
naip_path

'las-vegas-train-naip.tif'

数据下载后，我们将NAIP图像添加为栅格图层：

In [34]:
print(repr(naip_path))

'las-vegas-train-naip.tif'


In [38]:
m = leafmap.Map()
m.add_raster(naip_path, layer_name="NAIP Image")
m

InvalidURL: Failed to parse: http://::1:57111/api/metadata?&filename=d%3A%5Csoft%5Cgeo_program%5CI-GUIDE-GeoAI-Education%5Cnotebooks%5Clas-vegas-train-naip.tif

对于单波段栅格，您可以使用`vmin`和`vmax`指定颜色映射和值范围。这里我们添加地面高度栅格，将显示上限设置为10米，使建筑规模的结构突出：

In [39]:
m.add_raster(
    hag_path, vmin=0, vmax=10, colormap="terrain", layer_name="Height Above Ground"
)

InvalidURL: Failed to parse: http://::1:57111/api/metadata?&filename=d%3A%5Csoft%5Cgeo_program%5CI-GUIDE-GeoAI-Education%5Cnotebooks%5Clas-vegas-train-hag.tif

`add_raster()`接受本地文件路径和远程URL。使用图层控制在图层之间切换。

### 云优化GeoTIFF（COG）

云优化GeoTIFF允许您直接从远程服务器流式传输栅格数据，而无需下载整个文件。

In [19]:
m = leafmap.Map()
m.add_cog_layer(naip_url, name="Las Vegas NAIP")
m

Map(center=[-115.28050830697481, 36.29026032766879], controls=(ZoomControl(options=['position', 'zoom_in_text'…

当处理托管在Planetary Computer或AWS Open Data等云平台上的大型档案时，COG特别有用。

### 波段组合

`add_raster()`中的`indexes`参数允许您选择要显示的波段及其顺序。将NIR波段放在红色通道中使植被呈现亮红色，有助于区分植被区域和不透水表面：

In [20]:
m = leafmap.Map()
m.add_raster(naip_path, indexes=[4, 1, 2], layer_name="False Color")
m

InvalidURL: Failed to parse: http://::1:57111/api/metadata?&filename=d%3A%5Csoft%5Cgeo_program%5CI-GUIDE-GeoAI-Education%5Cnotebooks%5Clas-vegas-train-naip.tif

常见的NAIP波段组合包括：

- **真彩色**（波段1、2、3）：自然外观
- **假彩色**（波段4、1、2）：植被呈现亮红色，有助于区分建筑和植被

## 可视化Planetary Computer的数据

Microsoft Planetary Computer托管PB级的地理空间数据，可通过STAC API访问。`geoai`库提供了便捷函数来搜索、可视化和下载此目录中的数据。

### 浏览可用集合

`pc_collection_list()`函数将所有可用集合作为DataFrame检索。

In [21]:
collections = geoai.pc_collection_list()
print(f"Total collections: {len(collections)}")
collections.head(10)

Total collections: 134


,id,title,description
41,3dep-lidar-classification,USGS 3DEP Lidar Classification,This collection is derived from the [USGS 3DEP...
28,3dep-lidar-copc,USGS 3DEP Lidar Point Cloud,This collection contains source data from the ...
3,3dep-lidar-dsm,USGS 3DEP Lidar Digital Surface Model,This collection is derived from the [USGS 3DEP...
42,3dep-lidar-dtm,USGS 3DEP Lidar Digital Terrain Model,This collection is derived from the [USGS 3DEP...
40,3dep-lidar-dtm-native,USGS 3DEP Lidar Digital Terrain Model (Native),This collection is derived from the [USGS 3DEP...
19,3dep-lidar-hag,USGS 3DEP Lidar Height above Ground,This COG type is generated using the Z dimensi...
24,3dep-lidar-intensity,USGS 3DEP Lidar Intensity,This collection is derived from the [USGS 3DEP...
25,3dep-lidar-pointsourceid,USGS 3DEP Lidar Point Source,This collection is derived from the [USGS 3DEP...
31,3dep-lidar-returns,USGS 3DEP Lidar Returns,This collection is derived from the [USGS 3DEP...
2,3dep-seamless,USGS 3DEP Seamless DEMs,U.S.-wide digital elevation data at horizontal...


### 搜索STAC项目

使用`pc_stac_search()`查找符合您的空间和时间标准的项目。这里我们搜索覆盖巴尔的摩的NAIP图像：

In [22]:
naip_items = geoai.pc_stac_search(
    collection="naip",
    bbox=[-76.6657, 39.2648, -76.6478, 39.2724],
    time_range="2013-01-01/2014-12-31",
)
naip_items

[<Item id=md_m_3907643_se_18_1_20130917_20131112>]

```text
找到1个符合搜索条件的项目
[<Item id=md_m_3907643_se_18_1_20130917_20131112>]
```

您可以列出任何项目的可用资产：

In [ ]:
geoai.pc_item_asset_list(naip_items[0])

```text
['image', 'metadata', 'thumbnail', 'tilejson', 'rendered_preview']
```

### 可视化NAIP图像

`view_pc_item()`函数通过流式传输瓦片在交互式地图上渲染STAC项目，无需下载。

In [ ]:
geoai.view_pc_item(item=naip_items[0])

### 可视化土地覆盖数据

这里我们搜索切萨皮克湾高分辨率土地覆盖数据集，并使用分类颜色映射显示它：

In [ ]:
lc_items = geoai.pc_stac_search(
    collection="chesapeake-lc-13",
    bbox=[-76.6657, 39.2648, -76.6478, 39.2724],
    time_range="2013-01-01/2014-12-31",
    max_items=10,
)
lc_items

In [ ]:
geoai.view_pc_item(item=lc_items[0], colormap_name="tab10", basemap="SATELLITE")

像土地覆盖这样的分类数据集与`"tab10"`或`"Set3"`等定性颜色映射配合得很好。

### 可视化Landsat图像

对于像Landsat这样的多光谱数据，您可以选择特定波段或即时计算波段数学：

In [ ]:
landsat_items = geoai.pc_stac_search(
    collection="landsat-c2-l2",
    bbox=[-76.6657, 39.2648, -76.6478, 39.2724],
    time_range="2024-10-27/2024-12-31",
    query={"eo:cloud_cover": {"lt": 1}},
    max_items=10,
)
landsat_items

In [ ]:
geoai.pc_item_asset_list(landsat_items[0])

使用红、绿、蓝波段的真彩色合成：

In [ ]:
geoai.view_pc_item(item=landsat_items[0], assets=["red", "green", "blue"])

假彩色合成将近红外波段放在红色通道中，使植被呈现亮红色：

In [ ]:
geoai.view_pc_item(item=landsat_items[0], assets=["nir08", "red", "green"])

您也可以使用`expression`参数计算光谱指数：

In [ ]:
geoai.view_pc_item(
    item=landsat_items[0],
    expression="(nir08-red)/(nir08+red)",
    rescale="-1,1",
    colormap_name="greens",
    name="NDVI",
)

`expression`参数接受使用资产名称作为变量的波段数学运算，`rescale`控制值范围。计算在服务器端处理，因此不需要本地处理。

### 从Planetary Computer下载数据

使用`pc_stac_download()`将特定资产下载到磁盘：

In [ ]:
geoai.pc_stac_download(naip_items, output_dir="data", assets=["image", "thumbnail"])

您也可以直接将资产读取到xarray DataArray中而不保存：

In [ ]:
ds = geoai.read_pc_item_asset(lc_items[0], asset="data")
ds

## 在地图上处理矢量数据

矢量数据在GeoAI中用作训练标签、参考边界和模型输出。`leafmap`提供了灵活的方法来添加来自本地文件、远程URL和内存中GeoDataFrame的矢量图层。

### 添加GeoJSON和GeoDataFrame

您可以在工作流程的任何阶段可视化来自GeoJSON文件、URL或GeoDataFrame的矢量数据。

In [ ]:
import geopandas as gpd

gdf = gpd.read_file(buildings_path)
print(f"Features: {len(gdf)}")
gdf.head()

In [ ]:
m = leafmap.Map()
m.add_raster(naip_path)
m.add_gdf(gdf, layer_name="Building Footprints", zoom_to_layer=True)
m

您也可以直接从URL添加GeoJSON，而无需先将其加载到GeoDataFrame中：

In [ ]:
m = leafmap.Map()
m.add_raster(naip_path)
m.add_geojson(buildings_path, layer_name="Buildings", zoom_to_layer=True)
m

### 样式化矢量图层

自定义样式有助于区分要素类型并突出显示重要模式。您可以控制填充颜色、轮廓颜色、线宽和不透明度：

In [ ]:
style = {
    "color": "red",
    "weight": 2,
    "fillColor": "yellow",
    "fillOpacity": 0.3,
}
m = leafmap.Map()
m.add_raster(naip_path)
m.add_gdf(gdf, layer_name="Styled Buildings", style=style, zoom_to_layer=True)
m

`style`字典遵循Leaflet路径选项约定：`color`（轮廓）、`weight`（轮廓宽度）、`fillColor`和`fillOpacity`。

### 添加标记和点

对于点数据，您可以向地图添加标记。对于密集点图层，考虑使用标记聚类以保持可读性。

In [ ]:
m = leafmap.Map(center=[36.1617, -115.1524], zoom=11)
m.add_marker(location=[36.1617, -115.1524])
m

## 用于比较的分割面板地图

分割面板地图提供了一种结构化的方式来比较数据集——例如不同日期的图像或预测与地面真实情况——而不会丢失空间背景。

### 并排比较

`split_map()`方法创建一个在两个图层之间有可拖动分隔符的地图。这里我们将NAIP图像作为假彩色合成与地面高度栅格进行比较：

In [ ]:
m = leafmap.Map()
m.split_map(
    left_layer=naip_path,
    right_layer=hag_path,
    left_args={"indexes": [4, 1, 2]},
    right_args={"vmin": 0, "vmax": 10, "cmap": "terrain"},
    left_label="NAIP",
    right_label="HAG",
)
m

左右拖动滑块以显示每个图层，并查看图像中的高大结构如何对应于高HAG值。

## 可视化模型结果

模型输出的视觉检查揭示了模型失败的位置和原因，揭示了汇总统计无法捕捉的空间误差模式。

### 叠加预测

将预测栅格或矢量输出叠加在源图像上，使用部分透明度来评估突出显示的特征是否对应于真实对象：

In [ ]:
m = leafmap.Map()
m.add_raster(naip_path, layer_name="NAIP Imagery")
m.add_raster(mask_path, opacity=0.8, nodata=0, layer_name="Building Mask")
m

将`opacity`设置为1以下使图像显示出来，`nodata=0`使非建筑像素透明。

### 比较标签与源图像

分割面板地图也可用于在比较预测之前验证参考标签是否与源图像对齐：

In [ ]:
import leafmap

m = leafmap.Map()
m.split_map(
    left_layer=buildings_path,
    right_layer=naip_path,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
)
m

`geoai`库还提供了`create_split_map()`，在两个面板下方添加底图：

In [ ]:
geoai.create_split_map(
    left_layer=buildings_path,
    right_layer=naip_path,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
    basemap=naip_path,
)

拖动滑块以检查标注是否准确追踪建筑边界、是否遗漏了建筑，或者是否错误标记了非建筑结构。

## GeoAI可视化的最佳实践

**为您的数据类型选择合适的颜色映射。**像`"viridis"`或`"terrain"`这样的连续颜色映射适用于海拔或NDVI等连续数据。像`"RdBu"`这样的发散颜色映射更适用于具有有意义中心点的数据，例如温度异常或变化值。分类颜色映射应为每个类别使用明显、易于区分的颜色。避免使用彩虹颜色映射，因为它们缺乏自然的感知顺序，可能会产生误导。

**始终包含空间背景。**没有底图或参考要素的预测地图很难解释。包括卫星图像、行政边界或其他参考图层，帮助查看者定位并理解结果的地理环境。

**在比较中使用一致的样式。**当比较不同实验或时间段的模型输出时，对所有图层使用相同的颜色映射、值范围和不透明度设置。不一致的样式可能会产生不反映实际数据差异的虚假视觉差异。

**清晰地标注您的图层。**为每个图层提供一个出现在图层控制中的描述性名称。当您同时激活多个图层时，像"Building Predictions (U-Net)"这样的名称远比"Layer 1"有用。

**考虑您的受众。**技术同事可能会欣赏具有完整交互性的详细多层地图，而决策者可能需要更简单的可视化，带有清晰的图例和注释。根据使用您可视化的人群调整可视化的复杂性和格式。

## 关键要点

1. 交互式地图揭示了汇总统计无法捕捉的空间模式，使可视化在GeoAI生命周期中至关重要。
2. `leafmap`提供了一个高级Python接口，用于在Jupyter笔记本中创建交互式地理空间地图，内置支持底图、栅格图层和矢量叠加。
3. 栅格可视化通过`add_raster()`支持本地GeoTIFF，通过`add_cog_layer()`支持远程COG，具有可自定义的颜色映射、波段组合和值范围。
4. 矢量可视化处理GeoJSON文件、GeoDataFrame和点标记，具有灵活的样式，用于在图像上叠加标注。
5. 分割面板地图通过可拖动滑块实现并排比较，用于评估预测、比较图像和验证标注。
6. 模型结果叠加使用`nodata`实现透明度，使用`opacity`实现图层混合，将预测栅格与源图像结合。
7. 通过`geoai`集成Planetary Computer，您可以搜索、预览和下载云托管数据集，并在服务器端渲染波段组合和光谱指数。
8. 一致的样式和清晰的标签提高了可视化的可读性和有效性。